In [ ]:
import plotly.express as px
import polars as pl

from fryer import all as fryer

In [ ]:
df_postcode = fryer.data.uk_gov_ons_postcode_directory.read().select(
    pl.col("postcode"),
    pl.col("longitude"),
    pl.col("latitude"),
)
df_postcode.tail().collect()

In [ ]:
df = fryer.data.uk_gov_compare_school_performance.read_raw(year=2023).join(
    df_postcode, on="postcode", how="left"
)

df.head().collect()

In [ ]:
# Postcodes where the join has failed
df.filter(pl.col("longitude").is_null()).collect()["postcode"].value_counts().sort(
    "count", descending=True
)

In [ ]:
for col in (
    "status",
    "group",
    "funding_type",
    "ofsted_rating",
    "religious_character",
    "gender",
):
    df_len = df.group_by([col]).len().collect()
    display(
        df_len.pipe(
            px.bar,
            x=col,
            y="len",
            color=col,
            title=col,
            category_orders={
                col: df_len.sort(by="len", descending=True)[col].to_list(),
            },
        ),
    )

In [ ]:
df_interested = df.filter(pl.col("status") == "Open").collect()

df_interested.columns

In [ ]:
london_map = fryer.map.london()

df_primary = (
    df_interested.filter(pl.col("is_primary"))
    .drop_nulls(subset=["latitude", "longitude"])
    .with_columns(
        pl.when(
            (~pl.col("ofsted_rating").is_in(["Good", "Outstanding"]))
            .or_(pl.col("gender").is_in(["Boys"]))
            .or_(
                ~pl.col("funding_type").is_in(
                    ["State-funded primary", "State-funded secondary"]
                )
            )
        )
        .then(pl.lit(-1))
        .when(pl.col("ofsted_rating").is_in(["Good"]))
        .then(pl.lit(1))
        .when(pl.col("ofsted_rating").is_in(["Outstanding"]))
        .then(pl.lit(2))
        .alias("score")
    )
    .with_columns(
        # https://stackoverflow.com/a/41993318/16255028
        pl.col("score")
        .replace_strict({0: "red", 1: "orange", 2: "green"}, default=0)
        .alias("color"),
        pl.selectors.string().fill_null("NA"),
        pl.selectors.numeric().fill_null(float("nan")),
    )
    .with_columns(
        (
            pl.col("ofsted_rating")
            + " @ "
            + pl.col("date_last_ofsted_inspection").dt.strftime("%Y-%m-%d")
        ).alias("ofsted"),
        (
            "low: "
            + pl.col("age_low").cast(pl.String)
            + ", high: "
            + pl.col("age_high").cast(pl.String)
        ).alias("age"),
    )
    # .with_columns(
    #     (
    #         pl.col("tooltip")
    #         + "<br>Admissions Policy: "
    #         + pl.col("admissions_policy")
    #         + "<br>Number of Pupils: "
    #         + pl.col("number_of_pupils").cast(pl.String)
    #         + "<br>Girls: "
    #         + pl.col("percent_girls").round(2).cast(pl.String)
    #         + "<br>Age Low: "
    #         + pl.col("age_low").cast(pl.String)
    #         + ", Age High: "
    #         + pl.col("age_high").cast(pl.String)
    #         + "<br>Absence: "
    #         + pl.col("percent_absence").round(2).cast(pl.String)
    #         + ", Persistent Absence: "
    #         + pl.col("percent_persistent_absence").round(2).cast(pl.String)
    #         + "<br>First Language English: "
    #         + pl.col("percent_first_language_english").round(2).cast(pl.String)
    #         + "<br>Free School Meals: "
    #         + pl.col("percent_free_school_meals").round(2).cast(pl.String)
    #         + ", Last 6 Years: "
    #         + pl.col("percent_free_school_meals_last_6_years").round(2).cast(pl.String)
    #         + "<br>Education Health Care Plan: "
    #         + pl.col("percent_education_health_care_plan").round(2).cast(pl.String)
    #         + "<br>Special Education Needs Support: "
    #         + pl.col("percent_special_education_needs_support").round(2).cast(pl.String)
    #     ).alias("popup")
    # )
)
fryer.map.make_markers(
    df=df_primary,
    score="score",
    color="color",
    tooltip=(
        tooltip := ["name", "funding_type", "gender", "ofsted", "religious_character"]
    ),
    popup=[
        *tooltip,
        "admissions_policy",
        "number_of_pupils",
        "percent_girls",
        "age",
        "percent_absence",
        "percent_persistent_absence",
        "percent_first_language_english",
        "percent_free_school_meals",
        "percent_free_school_meals_last_6_years",
        "percent_education_health_care_plan",
        "percent_special_education_needs_support",
    ],
    cluster_threshold=(1, 1.2),
    max_cluster_radius=60,
    add_to_map=london_map,
)
london_map